In [2]:
from __future__ import annotations
from dataclasses import dataclass, field
from decimal import Decimal
from enum import Enum
from typing import Optional

In [3]:
@dataclass(frozen=True)
class Dinheiro:
    valor: Decimal

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError(f"Valor monetário não pode ser negativo: {self.valor}")

    def __add__(self, outro: Dinheiro) -> Dinheiro:
        return Dinheiro(self.valor + outro.valor)

    def __sub__(self, outro: Dinheiro) -> Dinheiro:
        resultado = self.valor - outro.valor
        return Dinheiro(max(resultado, Decimal("0")))

    def __mul__(self, fator: Decimal) -> Dinheiro:
        return Dinheiro((self.valor * fator).quantize(Decimal("0.01")))

    def __gt__(self, outro: Dinheiro) -> bool:
        return self.valor > outro.valor

    def __repr__(self):
        return f"R$ {self.valor:.2f}"

`__add__`— define o operador + entre dois Dinheiro. Retorna um novo objeto (imutabilidade):

In [4]:
d1 = Dinheiro(Decimal("10"))
d2 = Dinheiro(Decimal("5"))
d3 = d1 + d2  # Dinheiro(Decimal("15")) — novo objeto

In [10]:
@dataclass(frozen=True)
class Percentual:
    valor: Decimal  # 0 a 100

    def __post_init__(self):
        if not (Decimal("0") <= self.valor <= Decimal("100")):
            raise ValueError(f"Percentual inválido: {self.valor}")

    def aplicar_sobre(self, dinheiro: Dinheiro) -> Dinheiro:
        return dinheiro * (self.valor / Decimal("100"))

Encapsula a regra de que percentual só existe entre 0 e 100. O método aplicar_sobre converte o percentual em fator e usa o `__mul__` de Dinheiro:

In [11]:
p = Percentual(Decimal("10"))
p.aplicar_sobre(Dinheiro(Decimal("129.30")))  # R$ 12.93

R$ 12.93

Sem essa classe, você espalharia essa lógica de divisão por 100 em vários lugares.

`CodigoCupom`

In [8]:
@dataclass(frozen=True)
class CodigoCupom:
    valor: str

    def __post_init__(self):
        if not self.valor or len(self.valor) < 4:
            raise ValueError("Código de cupom deve ter ao menos 4 caracteres")
        object.__setattr__(self, "valor", self.valor.upper())

`object.__setattr__`— como a dataclass é `frozen=True`, normalmente não dá para modificar atributos nem dentro do `__post_init__`. A única forma de fazer isso é chamar o `__setattr__` do object diretamente, "bypassando" a proteção do frozen. Isso é necessário para normalizar o código para maiúsculo:

In [9]:
CodigoCupom("desconto10").valor  # "DESCONTO10"
CodigoCupom("AB")                # ValueError — menos de 4 chars

ValueError: Código de cupom deve ter ao menos 4 caracteres

`Quantidade`

In [12]:
@dataclass(frozen=True)
class Quantidade:
    valor: int

    def __post_init__(self):
        if self.valor < 0:
            raise ValueError("Quantidade não pode ser negativa")

    def subtrair(self, outra: Quantidade) -> Quantidade:
        if outra.valor > self.valor:
            raise ValueError("Estoque insuficiente")
        return Quantidade(self.valor - outra.valor)

    def esta_disponivel(self) -> bool:
        return self.valor > 0

Encapsula um `int` com regras de negócio. `subtrair` protege contra estoque negativo — a regra fica dentro do objeto, não espalhada em `ifs` pelo código:

In [13]:
Quantidade(5).subtrair(Quantidade(3))   # Quantidade(2)
Quantidade(2).subtrair(Quantidade(5))   # ValueError: Estoque insuficiente
Quantidade(0).esta_disponivel()         # False

ValueError: Estoque insuficiente

Entidades de Domínio

Diferente dos Value Objects, entidades têm identidade — dois produtos com o mesmo nome são o mesmo produto.